Summary of operations
1. Download prerequisite libraries
2. Perform data transformation and data segmentation
3. Define final features (animal classes) and NN architecture (convolutional)
4. Define loss function and optimiser
5. Train model
6. Export model parameters & calculate accuracy

In [3]:
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [4]:
transform = transforms.Compose([                             # Transformation pipeline for input images
    transforms.ToTensor(),                                   # Convert images to PyTorch tensors (tensors = multi-dim arrays)
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))   # Normalize tensors to speed up training/convergence
])

In [5]:
# Load CIFAR-10 dataset (part of torchvision.datasets)
train_data = torchvision.datasets.CIFAR10(root='./data',        # Directory to store/load dataset
                                          train=True,           # Load training dataset
                                          transform=transform,  # Apply the defined transformation pipeline above
                                          download=True)        # Download dataset if not already present

test_data = torchvision.datasets.CIFAR10(root='./data',         # Directory to store/load dataset
                                         train=False,           # Load test dataset, not training dataset
                                         transform=transform,   # Apply the defined transformation pipeline above
                                         download=True)         # Download dataset if not already present

# Use DataLoader to create iterable over the dataset with batching, shuffling, and parallel loading
train_loader = torch.utils.data.DataLoader(train_data,          # Dataset to load data from
                                            batch_size=32,      # Number of samples per batch
                                            shuffle=True,       # Shuffle data at every epoch (epoch = one full pass through dataset)
                                            num_workers=2)      # Number of subprocesses to use for data loading for parallelism

test_loader = torch.utils.data.DataLoader(test_data,            # Dataset to load data from
                                            batch_size=32,      # Number of samples per batch
                                            shuffle=False,      # Do not shuffle test data
                                            num_workers=2)      # Number of subprocesses to use for data loading for parallelism

Files already downloaded and verified
Files already downloaded and verified


In [6]:
# Take a look at a batch of training data
image, label = train_data[0]  # Get the first image and label from the training dataset
image.size()  # Check the size of the image tensor (3 color channels x 32 height x 32 width)

torch.Size([3, 32, 32])

In [7]:
# Define the classes in CIFAR-10 dataset
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [8]:
# Define convolutional neural network architecture
class NeuralNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Define convolutional layer, creating new shape of (12, 28, 28); ((32-5)/1+1) pixels with 12 feature maps
        self.conv1 = nn.Conv2d(in_channels=3,                        # 3 input color channels (RGB)
                               out_channels=12,                      # 12 output feature maps
                               kernel_size=5,)                       # 5x5 convolutional kernel
        # Define max pooling layer, reducing each feature map size from (28, 28) to (14, 14)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)            # Max pooling layer with 2x2 kernel and stride of 2
        # Define second convolutional layer, creating new shape of (24, 10, 10); ((14-5)/1+1) pixels with 24 feature maps
        self.conv2 = nn.Conv2d(in_channels=12,                       # 12 input
                                out_channels=24,                     # 24 output feature maps
                                kernel_size=5,)                      # 5x5 convolutional kernel
        
        # Define fully connected layer, mapping 24*5*5 input features to 120 features
        self.fcl = nn.Linear(in_features=24 * 5 * 5,                 # Fully connected layer with 24*5*5 input features
                             out_features=120)                       # and 120 output features
        # Define second fully connected layer, mapping 120 features to 84 features
        self.fc2 = nn.Linear(in_features=120,                        # Fully connected layer with 120 input features
                             out_features=84)                        # and 84 output features
        # Define output layer, mapping 84 features to 10 classes
        self.fc3 = nn.Linear(in_features=84,                         # Fully connected layer with 84 input features
                             out_features=10)                        # and 10 output features (one for each class)
        
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))               # Apply first convolutional layer with ReLU activation and pooling
        x = self.pool(F.relu(self.conv2(x)))               # Apply second convolutional layer with ReLU activation and pooling
        x = torch.flatten(x, 1)                            # Flatten the feature maps into a 1D feature vector
        x = F.relu(self.fcl(x))                            # Apply first fully connected layer with ReLU activation
        x = F.relu(self.fc2(x))                            # Apply second fully connected layer with ReLU activation
        x = self.fc3(x)                                    # Apply output layer
        return x

In [9]:
net = NeuralNet()                                  # Instantiate the neural network
loss_function = nn.CrossEntropyLoss()               # Define the loss function (used to measure prediction error)
optimizer = optim.SGD(net.parameters(),             # Define the optimizer (stochastic gradient descent with momentum)
                      lr=0.001,                     # Learning rate; controls step size during optimization
                      momentum=0.9)                 # Momentum factor to accelerate convergence

In [10]:
for epoch in range(30):                             # Loop over the dataset multiple times (10 epochs)
    print(f'Training epoch {epoch + 1}\n-------------------------------')

    running_loss = 0.0                              # Initialize loss accumulator for this epoch
    for i, data in enumerate(train_loader):
        inputs, labels = data                       # Get the inputs and labels from the current batch
        optimizer.zero_grad()                       # Zero/clear the parameter gradients before backward pass
        outputs = net(inputs)                       # Forward pass: pass inputs to network to get predictions (logits/scores)
        loss = loss_function(outputs, labels)       # Compute the loss between predictions and true labels
        loss.backward()                             # Backpropagate: compute gradient of loss wrt model parameters, tell parameters how to change to reduce loss
        optimizer.step()                            # Update model parameters based on computed gradients
        running_loss += loss.item()                 # Accumulate loss for reporting
    
    print(f'Epoch {epoch + 1} loss: {running_loss / len(train_loader):.4f}\n')

Training epoch 1
-------------------------------
Epoch 1 loss: 2.1567

Training epoch 2
-------------------------------
Epoch 2 loss: 1.7294

Training epoch 3
-------------------------------
Epoch 3 loss: 1.4894

Training epoch 4
-------------------------------
Epoch 4 loss: 1.3810

Training epoch 5
-------------------------------
Epoch 5 loss: 1.2886

Training epoch 6
-------------------------------
Epoch 6 loss: 1.2107

Training epoch 7
-------------------------------
Epoch 7 loss: 1.1421

Training epoch 8
-------------------------------
Epoch 8 loss: 1.0851

Training epoch 9
-------------------------------
Epoch 9 loss: 1.0362

Training epoch 10
-------------------------------
Epoch 10 loss: 0.9920

Training epoch 11
-------------------------------
Epoch 11 loss: 0.9540

Training epoch 12
-------------------------------
Epoch 12 loss: 0.9157

Training epoch 13
-------------------------------
Epoch 13 loss: 0.8763

Training epoch 14
-------------------------------
Epoch 14 loss: 0.84

In [11]:
torch.save(net.state_dict(), 'trained_net.pth')

In [12]:
net = NeuralNet()                                  # Instantiate the neural network
net.load_state_dict(torch.load('trained_net.pth')) # Load the trained model parameters

C:\Users\hieul\AppData\Local\Temp\ipykernel_35604\2855880047.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  net.load_state_dict(torch.load('trained_net.pth')) # Load th

<All keys matched successfully>

In [13]:
correct = 0
total = 0

net.eval()                                                # Set the network to evaluation mode
with torch.no_grad():
    for data in test_loader:                              # Iterate over the test dataset
        images, labels = data                             # Get the images and labels from the current batch
        outputs = net(images)                             # Forward pass: get predictions from the network
        _, predicted = torch.max(outputs.data, 1)         # Get the predicted class with the highest score
        total += labels.size(0)                           # Update total number of samples
        correct += (predicted == labels).sum().item()     # Update number of correct predictions

accuracy = 100 * correct / total                          # Calculate accuracy as percentage
print(f'Accuracy: {accuracy:.2f}%')


Accuracy: 68.67%


In [17]:
custom_transformation = transforms.Compose([
    transforms.Resize((32, 32)),                            # Resize images to 32x32 pixels
    transforms.ToTensor(),                                  # Convert images to PyTorch tensors
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize tensors
])

def load_and_preprocess_image(image_path):
    image = Image.open(image_path)                          # Load image from the specified path
    image = custom_transformation(image)                    # Apply the custom transformation pipeline
    image = image.unsqueeze(0)                              # Add batch dimension (1, 3, 32, 32) to make it compatible with the network input
    return image

image_paths = ['images/example1.jpg', 'images/example2.jpg']  # List of custom image file paths
images = [load_and_preprocess_image(path) for path in image_paths]  # Load and preprocess each image

net.eval()                                                # Set the network to evaluation mode
with torch.no_grad():
    for i, image in enumerate(images):
        outputs = net(image)                               # Forward pass: get predictions from the network
        _, predicted = torch.max(outputs.data, 1)         # Get the predicted class with the highest score
        print(f'Image {i + 1} predicted as: {classes[predicted.item()]}')  # Print the predicted class name

Image 1 predicted as: car
Image 2 predicted as: horse
